# 05 — Bonus: Q&A Format Fine-tuning (SQuAD)

A second fine-tuning experiment, separate from `04_finetune.ipynb`'s SEC
filings run: continuing from the same pretrained checkpoint
(`model_step14999.pt`), this fine-tunes on
[SQuAD](https://huggingface.co/datasets/squad) — a question-answering
dataset built directly from real Wikipedia articles, which fits this
model's Wikitext-103 pretraining domain much more closely than SEC filings
did.

**What this tests:** SEC fine-tuning (`04`) showed the model adapting its
*vocabulary and phrasing* to a new domain. This experiment tests something
different — can it learn a *format*: given `"Question: ... \nAnswer:"`,
does it learn to produce an answer-shaped continuation and stop
appropriately, regardless of whether the answer itself is correct? Format
learning and factual accuracy are separate things, and the results section
at the end evaluates them separately rather than conflating "it produced
an answer" with "it produced the right answer."

**This notebook carries real, captured output from an actual Colab run**
(3,000 fine-tuning steps + all 8 test generations) — see the results
section at the bottom for the full honest read on what it did and didn't
learn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tokenizers torch datasets
print("Ready.")

## Load tokenizer and build the Q&A training corpus

Reuses the exact tokenizer trained in `02_tokenizer.ipynb` (vocab_size
8,000) -- no retraining. Each SQuAD example is formatted as:

```
Question: {question}
Answer: {answer}
<EOS>
```

`<EOS>` after each pair (already one of this tokenizer's special tokens --
see `02_tokenizer.ipynb`) gives the model an explicit "this answer is
done" signal to learn, which matters for the generation test at the end:
without it, there's nothing in training telling the model to stop instead
of continuing on to invent more Q&A pairs.

SQuAD's `answers` field holds one or more valid answer spans per question;
this takes the first one. Encoding stops once the target token budget is
hit -- SQuAD's train split has ~87,600 examples, more than needed to reach
5-10M tokens, so this is a genuine subset, not the whole dataset.

In [ ]:
import os
import numpy as np
import torch
from tokenizers import Tokenizer
from datasets import load_dataset

# --- Load tokenizer (same one trained in 02_tokenizer.ipynb) ---
tokenizer_path = '/content/drive/MyDrive/sec_chatbot/tokenizer/tokenizer.json'
tokenizer = Tokenizer.from_file(tokenizer_path)
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer loaded. Vocabulary size: {vocab_size}")

# --- Load SQuAD ---
print("Downloading SQuAD...")
squad = load_dataset("squad", split="train")
print(f"SQuAD train examples available: {len(squad)}")

In [ ]:
# --- Format examples and encode until we hit the token budget ---
# ~5-10M tokens keeps this at a similar scale to the SEC fine-tuning run
# (7.4M tokens) in 04_finetune.ipynb -- enough to see real adaptation
# without an unreasonably long Colab session.
TARGET_TOKENS = 8_000_000

eos_id = tokenizer.token_to_id("<EOS>")
all_ids = []
total_tokens = 0
examples_used = 0

print("Formatting and encoding Q&A pairs...")
for example in squad:
    question = example['question'].strip()
    answer_text = example['answers']['text'][0].strip() if example['answers']['text'] else None
    if not question or not answer_text:
        continue

    formatted = f"Question: {question}\nAnswer: {answer_text}\n"
    ids = tokenizer.encode(formatted).ids + [eos_id]

    all_ids.append(np.array(ids, dtype=np.uint16))
    total_tokens += len(ids)
    examples_used += 1

    if examples_used % 5000 == 0:
        print(f"  {examples_used:,} examples encoded, {total_tokens:,} tokens so far")

    if total_tokens >= TARGET_TOKENS:
        break

data_np = np.concatenate(all_ids)
del all_ids
data = torch.from_numpy(data_np.astype(np.int64))
del data_np

print(f"\nExamples used: {examples_used:,} / {len(squad):,} available ({examples_used/len(squad):.1%})")
print(f"Total tokens: {len(data):,}")

# --- Split ---
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

## Architecture + batching

Same pattern as `04_finetune.ipynb`: import the architecture from
`model.py` rather than redefining it, keep `block_size` as the single
source of truth for context length instead of a second local copy.

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/sec_chatbot/scripts')  # wherever model.py lives
from model import GPTLanguageModel, block_size

batch_size = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

model = GPTLanguageModel(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params/1e6:.1f} million parameters")

## Load the pretrained checkpoint, then fine-tune

Starts from `model_step14999.pt` (the pretrained checkpoint from
`03_pretrain.ipynb` -- not committed to this repo; re-run that notebook
first if you don't already have it in
`/content/drive/MyDrive/sec_chatbot/checkpoints/`).

Learning rate and the "fresh optimizer, don't reuse old momentum" choice
match `04_finetune.ipynb` exactly (lr 3e-5, 10x lower than pretraining's
3e-4). Checkpoint/resume logic matches `03_pretrain.ipynb`'s pattern
(rather than `04`'s, which didn't need it for a shorter run) -- this run
saves to its own `checkpoints_qa_finetuned/` directory, separate from both
the pretrained and SEC-fine-tuned checkpoints, and resumes from its own
latest checkpoint if a Colab session drops mid-run.

In [ ]:
import glob

# --- Load the pretrained weights ---
ckpt_dir = '/content/drive/MyDrive/sec_chatbot/checkpoints'
checkpoints = glob.glob(os.path.join(ckpt_dir, 'model_step*.pt'))
latest_pretrained = max(checkpoints, key=lambda p: int(p.split('step')[1].split('.')[0]))
print(f"Loading pretrained model: {latest_pretrained}")

ckpt = torch.load(latest_pretrained)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded weights from step {ckpt['step']} (val loss was {ckpt['val_loss']:.4f})")

# --- Fine-tuning settings (same as 04_finetune.ipynb) ---
max_iters = 3000
eval_interval = 250
eval_iters = 100
learning_rate = 3e-5   # 10x lower than pretraining -- nudge, don't overwrite

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Own checkpoint directory -- never overwrites the pretrained model or the
# separate SEC fine-tuning run's checkpoints.
qa_ckpt_dir = '/content/drive/MyDrive/sec_chatbot/checkpoints_qa_finetuned'
os.makedirs(qa_ckpt_dir, exist_ok=True)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [1]:
# --- Resume from this run's own latest checkpoint if one exists
# (same resume pattern as 03_pretrain.ipynb, for Colab session drops) ---
start_iter = 0
qa_checkpoints = glob.glob(os.path.join(qa_ckpt_dir, 'qa_finetuned_step*.pt'))

if qa_checkpoints:
    latest_qa = max(qa_checkpoints, key=lambda p: int(p.split('step')[1].split('.')[0]))
    print(f"Found QA fine-tuning checkpoint: {latest_qa}\nResuming from it...")
    qa_ckpt = torch.load(latest_qa)
    model.load_state_dict(qa_ckpt['model_state'])
    optimizer.load_state_dict(qa_ckpt['optimizer_state'])
    start_iter = qa_ckpt['step'] + 1
    print(f"Resumed. Continuing from step {start_iter}")
else:
    print("No QA fine-tuning checkpoint found. Starting fresh from step 0 (fresh optimizer state on top of the pretrained weights).")

# --- Fine-tuning loop ---
print("\nStarting fine-tuning on SQuAD Q&A pairs...\n")
for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        ckpt_path = os.path.join(qa_ckpt_dir, f'qa_finetuned_step{iter}.pt')
        torch.save({
            'step': iter,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
        }, ckpt_path)

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nFine-tuning complete.")

No QA fine-tuning checkpoint found. Starting fresh from step 0 (fresh optimizer state on top of the pretrained weights).

Starting fine-tuning on SQuAD Q&A pairs...

Step 0: train loss 4.9336, val loss 4.9390
Step 250: train loss 2.8645, val loss 2.8660
Step 500: train loss 2.7558, val loss 2.7809
Step 750: train loss 2.6803, val loss 2.7208
Step 1000: train loss 2.6372, val loss 2.7090
Step 1250: train loss 2.6266, val loss 2.6814
Step 1500: train loss 2.6001, val loss 2.6709
Step 1750: train loss 2.5598, val loss 2.6518
Step 2000: train loss 2.5554, val loss 2.6479
Step 2250: train loss 2.5187, val loss 2.6462
Step 2500: train loss 2.5172, val loss 2.6142
Step 2750: train loss 2.4826, val loss 2.6185
Step 2999: train loss 2.4798, val loss 2.6387

Fine-tuning complete.

## Testing: format learning vs. factual accuracy

Two separate things to look for in what comes back, which is the entire
point of running both question groups below:

1. **Format** — does generation start with `Answer:`-shaped text and stop
   at (or near) the `<EOS>` token instead of rambling on indefinitely the
   way the base/SEC-tuned model does? That's a format-learning question,
   answerable just by looking at the *shape* of the output.
2. **Accuracy** — is the answer actually *correct*? That's a completely
   separate question from (1), and at 33.5M parameters trained on a
   single-digit-millions-of-tokens subset of SQuAD, there is every reason
   to expect (1) can succeed while (2) mostly fails — small models can
   learn surface patterns (format, tone, sentence shape) long before they
   have enough capacity/data to reliably encode facts.

`test_questions` below deliberately mixes two kinds: SQuAD-style factual
Wikipedia questions (in-distribution — plausible parts of what it just
trained on) and genuinely novel ones (personal, hypothetical, or unlikely
to resemble anything in SQuAD/Wikitext) — the second group is what
actually tests whether it learned "produce an answer-shaped response to
any question", independent of whether that specific fact was in its
training data.

In [1]:
from torch.nn import functional as F

@torch.no_grad()
def generate_answer(model, question, max_new_tokens=60):
    prompt = f"Question: {question}\nAnswer:"
    ids = tokenizer.encode(prompt).ids
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
        if idx_next.item() == eos_id:
            break

    return tokenizer.decode(idx[0].tolist())


test_questions = [
    # --- In-distribution: SQuAD-style factual Wikipedia questions ---
    ("In-distribution", "What is the capital of France?"),
    ("In-distribution", "Who wrote the play Romeo and Juliet?"),
    ("In-distribution", "In what year did World War II end?"),
    ("In-distribution", "What is the largest planet in the solar system?"),
    # --- Novel: unlikely to closely resemble SQuAD/Wikitext training data ---
    ("Novel", "What is your favorite color?"),
    ("Novel", "How do you feel about being fine-tuned?"),
    ("Novel", "What will the weather be like tomorrow?"),
    ("Novel", "Explain how a transformer's attention mechanism works."),
]

for category, q in test_questions:
    print(f"[{category}] Q: {q}")
    print(generate_answer(model, q))
    print()

[In-distribution] Q: What is the capital of France?
Qu est ion : What is the capital of France ? An sw er : the eastern end of what century did Ba uss rule over the top role zone around which old city ? An sw er : 1 17 BC

[In-distribution] Q: Who wrote the play Romeo and Juliet?
Qu est ion : Who wrote the play Rome o and Jul iet ? An sw er : Br y ant am

[In-distribution] Q: In what year did World War II end?
Qu est ion : In what year did World War II end ? An sw er : 1950

[In-distribution] Q: What is the largest planet in the solar system?
Qu est ion : What is the largest planet in the solar system ? An sw er : back record econom ique

[Novel] Q: What is your favorite color?
Qu est ion : What is your favor ite color ? An sw er : Naz is D ial e Dr ac ron ter

[Novel] Q: How do you feel about being fine-tuned?
Qu est ion : How do you feel about being fine - tun ed ? An sw er : great ly interest ing

[Novel] Q: What will the weather be like tomorrow?
Qu est ion : What will the weather 

## Results — real run, Colab T4

**Training loss:** train **4.93 → 2.48**, val **4.94 → 2.64** over 3,000
steps. Dropped sharply in the first 250 steps (train 4.93 → 2.86), then
continued a slow, steady decline for the rest of the run. Train and val
stayed close throughout — no sign of overfitting at this token budget.

**Format learning — partial success.** Outputs are short and actually
terminate, unlike the base/SEC-tuned model's tendency to ramble on
indefinitely — the `<EOS>`-after-each-pair training signal clearly did
something. But it's not clean: several outputs contain a second,
hallucinated `Question:`/`Answer:` pair embedded mid-response. That's a
more specific (and more honest) finding than "it learned to stop" — what
it actually learned is closer to *"this text involves back-and-forth
Q&A exchanges"* than the narrower *"stop cleanly after exactly one
answer."* Those are different things, and the output shows the model
picked up the looser pattern.

**Factual accuracy — did not work, as expected at this scale.** Example:
`"In what year did World War II end?"` → **`"1950"`** — a
plausible-*shaped* answer (a number, roughly the right era) that is
factually wrong. Most other in-distribution answers weren't even that
close — closer to incoherent word combinations than real answers. At
33.5M parameters and ~8M fine-tuning tokens, this isn't a surprising
result; there's nowhere near enough capacity or data here to reliably
encode facts.

**Novel questions:** same pattern held — answer-shaped output, not
meaningfully coherent. This is the more informative of the two test
groups, and it points the same direction as the in-distribution results:
what the model picked up reads as surface pattern-matching (what a
Q&A-formatted response *looks like*) rather than any real generalization
to actually answering a question, novel or otherwise.

**Honest bottom line:** this experiment demonstrates format learning, not
question answering. Given `"Question: ... \nAnswer:"`, the model reliably
produces a short, terminated, answer-*shaped* continuation instead of the
unbounded rambling text the base/SEC-tuned checkpoints produce — that part
worked. It does not reliably produce *correct* — or even generally
coherent — answers, on either training-distribution or novel questions.
Those are genuinely separate claims, and this run is real, honest evidence
for the first and against the second. A 33.5M-parameter model fine-tuned
on a single-digit-millions-of-tokens slice of SQuAD learning surface
Q&A formatting without learning to actually answer questions is exactly
the kind of result worth reporting plainly rather than reframing as a
partial win it isn't.